# Segunda fase

In [59]:
import pandas

# Importing classes of the project

# Reload classes in memory every time this code block is executed
%load_ext autoreload
%autoreload 2

from data_analysis.analizer import DataAnalizer, TripDataAnalizer
from model_building.KNN import KNN

import numpy as np
import pandas as pd

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [66]:
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("../out/dataset.csv")
del df["congestion_surcharge"]
del df["pickup_time_in_seconds"]
del df["dropoff_time_in_seconds"]


# =============================================
# 1. REGRESSION DATASET (continuous target)
# =============================================

# Separate features and target
X_reg = df.drop(columns=['fare_amount'])
y_reg = df['fare_amount']

# Scale only the features (not target)
scaler = StandardScaler()
X_reg_scaled = scaler.fit_transform(X_reg)

# Create scaled DataFrame for regression
df_regression = pd.DataFrame(X_reg_scaled, columns=X_reg.columns)
df_regression['fare_amount'] = y_reg.values  # Add unscaled target

# =============================================
# 2. CLASSIFICATION DATASET (categorical target)
# =============================================

# Create fare classes
bins = [-np.inf, 10, 30, 60, np.inf]
labels = [1, 2, 3, 4]

# Create classification target
df_classification = df.copy()
df_classification['fare_class'] = pd.cut(
    df['fare_amount'],
    bins=bins,
    labels=labels
)

# Separate features and target
X_clf = df_classification.drop(columns=['fare_amount', 'fare_class'])
y_clf = df_classification['fare_class']

# Scale features using SAME scaler (important for consistency)
X_clf_scaled = scaler.transform(X_clf)  # Use existing scaler

# Create scaled DataFrame for classification
df_classification_scaled = pd.DataFrame(X_clf_scaled, columns=X_clf.columns)
df_classification_scaled['fare_class'] = y_clf.values  # Add target

# =============================================
# Verification
# =============================================
print("Regression dataset:")
print(df_regression.head())

print("\nClassification dataset:")
print(df_classification_scaled.head())

print(df_classification['fare_class'].value_counts(normalize=True))

Regression dataset:
   trip_distance  tip_amount  tolls_amount     extra  passenger_count  \
0      -0.666681   -0.759165     -0.231572 -0.466372        -0.470065   
1      -0.405765    0.621948     -0.231572 -0.466372        -0.470065   
2      -0.456925   -0.172192     -0.231572 -0.466372        -0.470065   
3      -0.152522   -0.759165     -0.231572 -0.466372        -0.470065   
4       1.650870    0.967227     -0.231572 -0.466372        -0.470065   

   pickup_hour  pickup_day_of_week  pickup_day_of_month  pickup_month  \
0    -2.316087           -1.018236            -1.674579     -1.535088   
1    -2.316087           -1.018236            -1.674579     -1.535088   
2    -2.316087           -1.018236            -1.674579     -1.535088   
3    -2.316087           -1.018236            -1.674579     -1.535088   
4    -2.316087           -1.018236            -1.674579     -1.535088   

   dropoff_hour  dropoff_day_of_week  dropoff_day_of_month  dropoff_month  \
0     -2.286173          

In [64]:
# Regression Analysis of KNN
analizer_reg = DataAnalizer(df_regression, "fare_amount", test_size=0.05)

rmse_results = {}
for i in range(5, 26, 5):
    print(f"Training KNN Regression for k = {i}")
    knn_reg = KNN(i, problem_type="regression")
    knn_reg.fit(analizer_reg.data_train, analizer_reg.labels_train)
    pred_values = knn_reg.predict(analizer_reg.data_test)

    # Calculate Root Mean Squared Error
    rmse = np.sqrt(np.mean((pred_values - analizer_reg.labels_test)**2))
    rmse_results[i] = rmse
    print(f"k={i}: RMSE = {rmse:.2f}")

print("\nFinal Regression Results:")
print(rmse_results)

Data divided successfully.
Training KNN Regression for k = 5
Mode: Regression (continuous target)
k=5: RMSE = 5.48
Training KNN Regression for k = 10
Mode: Regression (continuous target)
k=10: RMSE = 5.62
Training KNN Regression for k = 15
Mode: Regression (continuous target)
k=15: RMSE = 5.69
Training KNN Regression for k = 20
Mode: Regression (continuous target)
k=20: RMSE = 5.78
Training KNN Regression for k = 25
Mode: Regression (continuous target)
k=25: RMSE = 5.85

Final Regression Results:
{5: np.float64(5.477072277027963), 10: np.float64(5.61534676324649), 15: np.float64(5.6902349968675034), 20: np.float64(5.782494103264503), 25: np.float64(5.85011939948596)}


In [65]:
# Classification Analysis of KNN
analizer_clf = DataAnalizer(df_classification_scaled, "fare_class", test_size=0.05)

precision_results = {}
for i in range(5, 26, 5):
    print(f"Training KNN Classification for k = {i}")
    knn_clf = KNN(i, problem_type="classification")
    knn_clf.fit(analizer_clf.data_train, analizer_clf.labels_train)
    pred_labels = knn_clf.predict(analizer_clf.data_test)

    # Calculate Precision (accuracy)
    precision = np.mean(pred_labels == analizer_clf.labels_test)
    precision_results[i] = precision
    print(f"k={i}: Precision = {precision:.4f}")

print("\nFinal Classification Results:")
print(precision_results)

Data divided successfully.
Training KNN Classification for k = 5
Mode: Classification (discrete target)
k=5: Precision = 0.7618
Training KNN Classification for k = 10
Mode: Classification (discrete target)
k=10: Precision = 0.7661
Training KNN Classification for k = 15
Mode: Classification (discrete target)
k=15: Precision = 0.7649
Training KNN Classification for k = 20
Mode: Classification (discrete target)
k=20: Precision = 0.7654
Training KNN Classification for k = 25
Mode: Classification (discrete target)
k=25: Precision = 0.7564

Final Classification Results:
{5: np.float64(0.7618147448015122), 10: np.float64(0.7660680529300568), 15: np.float64(0.7648865784499055), 20: np.float64(0.7653591682419659), 25: np.float64(0.7563799621928167)}
